# Post-Training Validation & Packaging

Self-contained notebook. Runs on Colab with a GPU.

**Prerequisites:** Training already completed and checkpoints saved to Google Drive.

Steps:
1. Mount Drive & configure paths
2. Train/test split (20% held-out)
3. Inference on **all** saved checkpoints
4. Compute metrics (MAE, MSE, RMSE, PSNR, SSIM, LPIPS)
5. Pick best checkpoint
6. Visual comparison grids
7. Comparison table vs old model
8. Package & zip deliverable

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import subprocess, sys
for pkg in ['scikit-image', 'opencv-python', 'pillow']:
    try:
        __import__(pkg.replace('-', '_'))
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

try:
    import lpips
    HAS_LPIPS = True
    print('LPIPS available')
except ImportError:
    HAS_LPIPS = False
    print('LPIPS not available (pip install lpips to enable)')

import torch
print(f'torch {torch.__version__}  cuda: {torch.cuda.is_available()}')

from skimage.metrics import structural_similarity as skimage_ssim
print('Dependencies OK')

LPIPS not available (pip install lpips to enable)
torch 2.10.0+cu128  cuda: True
Dependencies OK


In [3]:
# ===== EDIT THESE PATHS =====
DRIVE_ROOT = '/content/drive/MyDrive/Cobas'

# Directory where training ran (contains checkpoints/ and samples/)
WORK_DIR = f'{DRIVE_ROOT}/gan_run'

# Preprocessed frame folders (256x256 PNG)
OPTICAL_FRAMES = f'{DRIVE_ROOT}/cobas/preprocessed/opt'
THERMAL_FRAMES = f'{DRIVE_ROOT}/cobas/preprocessed/therm'
# =============================

import os
for label, p in [('WORK_DIR', WORK_DIR), ('OPTICAL_FRAMES', OPTICAL_FRAMES), ('THERMAL_FRAMES', THERMAL_FRAMES)]:
    exists = os.path.exists(p)
    print(f'{label}: {p} -> {"OK" if exists else "MISSING"}')
    if not exists:
        raise FileNotFoundError(f'{label} not found: {p}')

# List available checkpoints
ckpt_dir = os.path.join(WORK_DIR, 'checkpoints')
ckpts = sorted([f for f in os.listdir(ckpt_dir) if f.startswith('epoch_') and f.endswith('.pt')])
print(f'\nCheckpoints ({len(ckpts)}):')
for c in ckpts:
    size_mb = os.path.getsize(os.path.join(ckpt_dir, c)) / 1024 / 1024
    print(f'  {c}  ({size_mb:.1f} MB)')

WORK_DIR: /content/drive/MyDrive/Cobas/gan_run -> OK
OPTICAL_FRAMES: /content/drive/MyDrive/Cobas/cobas/preprocessed/opt -> OK
THERMAL_FRAMES: /content/drive/MyDrive/Cobas/cobas/preprocessed/therm -> OK

Checkpoints (4):
  epoch_0005.pt  (323.6 MB)
  epoch_0010.pt  (323.6 MB)
  epoch_0015.pt  (323.6 MB)
  epoch_0020.pt  (323.6 MB)


In [4]:
import random, shutil, json, math, time, csv
from pathlib import Path
import numpy as np
from PIL import Image
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import make_grid, save_image

IMG_SIZE = 256
N_RES = 9  # 9 residual blocks for 256x256
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
TEST_RATIO = 0.20

print(f'Device: {DEVICE}')

class ConvBlock(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=1, p=1, use_norm=True, use_relu=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, k, s, p, bias=not use_norm)]
        if use_norm:
            layers.append(nn.InstanceNorm2d(out_c))
        if use_relu:
            layers.append(nn.ReLU(inplace=True))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class DeconvBlock(nn.Module):
    def __init__(self, in_c, out_c, k=3, s=2, p=1, out_p=1, use_norm=True, use_relu=True):
        super().__init__()
        layers = [nn.ConvTranspose2d(in_c, out_c, k, s, p, out_p, bias=not use_norm)]
        if use_norm:
            layers.append(nn.InstanceNorm2d(out_c))
        if use_relu:
            layers.append(nn.ReLU(inplace=True))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block = nn.Sequential(
            nn.ReflectionPad2d(1),
            nn.Conv2d(c, c, 3, 1, 0, bias=False),
            nn.InstanceNorm2d(c),
            nn.ReLU(inplace=True),
            nn.ReflectionPad2d(1),
            nn.Conv2d(c, c, 3, 1, 0, bias=False),
            nn.InstanceNorm2d(c),
        )
    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self, in_nc=3, out_nc=3, n_res=9):
        super().__init__()
        layers = [
            nn.ReflectionPad2d(3),
            nn.Conv2d(in_nc, 64, 7, 1, 0, bias=False),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
            ConvBlock(64, 128, k=3, s=2, p=1),
            ConvBlock(128, 256, k=3, s=2, p=1),
        ]
        for _ in range(n_res):
            layers.append(ResBlock(256))
        layers += [
            DeconvBlock(256, 128, k=3, s=2, p=1, out_p=1),
            DeconvBlock(128, 64, k=3, s=2, p=1, out_p=1),
            nn.ReflectionPad2d(3),
            nn.Conv2d(64, out_nc, 7, 1, 0),
            nn.Tanh(),
        ]
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

def denorm(t):
    return (t.clamp(-1, 1) + 1.0) * 0.5

def to_np(t):
    return t.detach().cpu().permute(1, 2, 0).numpy().astype(np.float32).clip(0, 1)

tf_eval = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE), interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

print('Model architecture defined.')

Device: cuda
Model architecture defined.


In [5]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

WORK = Path(WORK_DIR)
OPT_DIR = Path(OPTICAL_FRAMES)
THM_DIR = Path(THERMAL_FRAMES)

opt_files = sorted([p for p in OPT_DIR.iterdir() if p.suffix.lower() in IMG_EXTS and p.is_file()])
thm_files = sorted([p for p in THM_DIR.iterdir() if p.suffix.lower() in IMG_EXTS and p.is_file()])
N = min(len(opt_files), len(thm_files))
print(f'Total paired frames: {N}')

indices = list(range(N))
rng = random.Random(SEED)
rng.shuffle(indices)
n_test = max(1, int(round(N * TEST_RATIO)))
test_idx = sorted(indices[:n_test])
train_idx = sorted(indices[n_test:])
print(f'Train: {len(train_idx)}  Test: {len(test_idx)}')

results_root = WORK / 'results' / 'cobas_clean_v1'
test_out = results_root / 'test_outputs'
for d in [test_out / 'real_A', test_out / 'fake_B', test_out / 'real_B', test_out / 'grids']:
    d.mkdir(parents=True, exist_ok=True)
print(f'Results dir: {results_root}')
print(f'Test sample indices: {test_idx[:5]}...{test_idx[-3:]}')

Total paired frames: 407
Train: 326  Test: 81
Results dir: /content/drive/MyDrive/Cobas/gan_run/results/cobas_clean_v1
Test sample indices: [2, 9, 11, 26, 29]...[396, 398, 406]


In [6]:
ckpt_dir_path = WORK / 'checkpoints'
ckpt_paths = sorted(ckpt_dir_path.glob('epoch_*.pt'))
print(f'Evaluating {len(ckpt_paths)} checkpoints on {len(test_idx)} test frames...')

if HAS_LPIPS:
    lpips_fn = lpips.LPIPS(net='alex').to(DEVICE).eval()

all_results = {}

for ckpt_path in ckpt_paths:
    tag = ckpt_path.stem
    print(f'\n--- {tag} ---')

    payload = torch.load(ckpt_path, map_location='cpu')
    G = Generator(3, 3, n_res=N_RES).to(DEVICE)
    G.load_state_dict(payload['G_o2t'])
    G.eval()

    mae_l, mse_l, rmse_l, psnr_l, ssim_l, lpips_l = [], [], [], [], [], []

    ckpt_out = test_out / tag
    ckpt_out.mkdir(exist_ok=True)

    t0 = time.time()
    with torch.no_grad():
        for i in test_idx:
            o_img = Image.open(opt_files[i]).convert('RGB')
            t_img = Image.open(thm_files[i]).convert('RGB')
            o_t = tf_eval(o_img).unsqueeze(0).to(DEVICE)
            t_t = tf_eval(t_img).unsqueeze(0).to(DEVICE)

            fake_t = G(o_t)

            pred_01 = denorm(fake_t).clamp(0, 1)
            tgt_01 = denorm(t_t).clamp(0, 1)
            inp_01 = denorm(o_t).clamp(0, 1)

            p_np = to_np(pred_01[0])
            t_np = to_np(tgt_01[0])

            diff = p_np - t_np
            mae_v = float(np.mean(np.abs(diff)))
            mse_v = float(np.mean(diff ** 2))
            rmse_v = math.sqrt(mse_v)
            psnr_v = 20 * math.log10(1.0) - 10 * math.log10(max(mse_v, 1e-12))
            ssim_v = float(skimage_ssim(t_np, p_np, data_range=1.0, channel_axis=2))

            mae_l.append(mae_v)
            mse_l.append(mse_v)
            rmse_l.append(rmse_v)
            psnr_l.append(psnr_v)
            ssim_l.append(ssim_v)

            if HAS_LPIPS:
                lpips_l.append(float(lpips_fn(fake_t, t_t).item()))

            fname = f'{i:06d}'
            save_image(inp_01[0], str(test_out / 'real_A' / f'{fname}.png'))
            save_image(pred_01[0], str(test_out / 'fake_B' / f'{fname}.png'))
            save_image(tgt_01[0], str(test_out / 'real_B' / f'{fname}.png'))

            save_image(inp_01[0], str(ckpt_out / f'{fname}_input.png'))
            save_image(pred_01[0], str(ckpt_out / f'{fname}_pred.png'))
            save_image(tgt_01[0], str(ckpt_out / f'{fname}_target.png'))

    elapsed = time.time() - t0
    throughput = len(test_idx) / max(elapsed, 1e-9)

    res = {
        'checkpoint': tag,
        'n_test': len(test_idx),
        'MAE_mean': float(np.mean(mae_l)),
        'MSE_mean': float(np.mean(mse_l)),
        'RMSE_mean': float(np.mean(rmse_l)),
        'PSNR_mean': float(np.mean(psnr_l)),
        'PSNR_std': float(np.std(psnr_l)),
        'SSIM_mean': float(np.mean(ssim_l)),
        'SSIM_std': float(np.std(ssim_l)),
        'elapsed_sec': elapsed,
        'throughput_img_s': throughput,
    }
    if HAS_LPIPS:
        res['LPIPS_mean'] = float(np.mean(lpips_l))
        res['LPIPS_std'] = float(np.std(lpips_l))

    all_results[tag] = res
    lpips_str = f'  LPIPS={res["LPIPS_mean"]:.4f}' if HAS_LPIPS else ''
    print(f'  PSNR={res["PSNR_mean"]:.2f} ± {res["PSNR_std"]:.2f}  '
          f'SSIM={res["SSIM_mean"]:.4f} ± {res["SSIM_std"]:.4f}  '
          f'MAE={res["MAE_mean"]:.4f}  RMSE={res["RMSE_mean"]:.4f}  '
          f'({throughput:.1f} img/s){lpips_str}')

print('\n=== All checkpoints evaluated ===')

Evaluating 4 checkpoints on 81 test frames...

--- epoch_0005 ---
  PSNR=21.82 ± 1.65  SSIM=0.7521 ± 0.0209  MAE=0.0468  RMSE=0.0825  (0.9 img/s)

--- epoch_0010 ---
  PSNR=23.49 ± 2.92  SSIM=0.8059 ± 0.0365  MAE=0.0357  RMSE=0.0703  (3.4 img/s)

--- epoch_0015 ---
  PSNR=22.72 ± 1.83  SSIM=0.8302 ± 0.0229  MAE=0.0342  RMSE=0.0747  (3.4 img/s)

--- epoch_0020 ---
  PSNR=26.27 ± 3.31  SSIM=0.8766 ± 0.0323  MAE=0.0256  RMSE=0.0519  (3.4 img/s)

=== All checkpoints evaluated ===


In [7]:
if not all_results:
    raise RuntimeError('No results. Run Cell 6 first.')

ranked = sorted(all_results.items(), key=lambda kv: (-kv[1]['SSIM_mean'], -kv[1]['PSNR_mean']))

hdr = f'{"Checkpoint":<20} {"PSNR":>8} {"SSIM":>8} {"MAE":>8} {"RMSE":>8}'
if HAS_LPIPS:
    hdr += f' {"LPIPS":>8}'
print(hdr)
print('-' * len(hdr))
for tag, r in ranked:
    line = f'{tag:<20} {r["PSNR_mean"]:>8.2f} {r["SSIM_mean"]:>8.4f} {r["MAE_mean"]:>8.4f} {r["RMSE_mean"]:>8.4f}'
    if HAS_LPIPS:
        line += f' {r["LPIPS_mean"]:>8.4f}'
    print(line)

best_tag, best_res = ranked[0]
print(f'\n>>> Best: {best_tag}  (PSNR={best_res["PSNR_mean"]:.2f}  SSIM={best_res["SSIM_mean"]:.4f})')

Checkpoint               PSNR     SSIM      MAE     RMSE
--------------------------------------------------------
epoch_0020              26.27   0.8766   0.0256   0.0519
epoch_0015              22.72   0.8302   0.0342   0.0747
epoch_0010              23.49   0.8059   0.0357   0.0703
epoch_0005              21.82   0.7521   0.0468   0.0825

>>> Best: epoch_0020  (PSNR=26.27  SSIM=0.8766)


In [8]:
best_ckpt_path = WORK / 'checkpoints' / f'{best_tag}.pt'
payload = torch.load(best_ckpt_path, map_location='cpu')
G_best = Generator(3, 3, n_res=N_RES).to(DEVICE)
G_best.load_state_dict(payload['G_o2t'])
G_best.eval()

grid_dir = test_out / 'grids'
grid_dir.mkdir(parents=True, exist_ok=True)

per_sample_rows = []
with torch.no_grad():
    for i in test_idx:
        o_img = Image.open(opt_files[i]).convert('RGB')
        t_img = Image.open(thm_files[i]).convert('RGB')
        o_t = tf_eval(o_img).unsqueeze(0).to(DEVICE)
        t_t = tf_eval(t_img).unsqueeze(0).to(DEVICE)
        fake_t = G_best(o_t)

        inp_01 = denorm(o_t).clamp(0, 1)
        pred_01 = denorm(fake_t).clamp(0, 1)
        tgt_01 = denorm(t_t).clamp(0, 1)

        row = torch.cat([inp_01[0], pred_01[0], tgt_01[0]], dim=2)
        per_sample_rows.append(row)
        save_image(row, str(grid_dir / f'{i:06d}_comparison.png'))

print(f'Saved {len(per_sample_rows)} per-sample grids')

# Summary grid: 12 evenly-spaced samples (optical | pred | thermal side by side)
N_SUMMARY = min(12, len(per_sample_rows))
sel = np.linspace(0, len(per_sample_rows) - 1, N_SUMMARY, dtype=int)
summary = make_grid(torch.stack([per_sample_rows[j] for j in sel]), nrow=1, padding=4, pad_value=1.0)
save_image(summary, str(grid_dir / 'summary_grid.png'))

# 3-row layout: all inputs / all preds / all targets
N_WIDE = min(12, len(test_idx))
wide_sel = np.linspace(0, len(test_idx) - 1, N_WIDE, dtype=int)
all_inp, all_pred, all_tgt = [], [], []
with torch.no_grad():
    for j in wide_sel:
        i = test_idx[j]
        o_t = tf_eval(Image.open(opt_files[i]).convert('RGB')).unsqueeze(0).to(DEVICE)
        t_t = tf_eval(Image.open(thm_files[i]).convert('RGB')).unsqueeze(0).to(DEVICE)
        fake_t = G_best(o_t)
        all_inp.append(denorm(o_t).clamp(0, 1)[0])
        all_pred.append(denorm(fake_t).clamp(0, 1)[0])
        all_tgt.append(denorm(t_t).clamp(0, 1)[0])

inp_g = make_grid(torch.stack(all_inp), nrow=4, padding=2, pad_value=1.0)
pred_g = make_grid(torch.stack(all_pred), nrow=4, padding=2, pad_value=1.0)
tgt_g = make_grid(torch.stack(all_tgt), nrow=4, padding=2, pad_value=1.0)

comp = torch.cat([inp_g, pred_g, tgt_g], dim=1)
save_image(comp, str(grid_dir / 'comparison_grid.png'))

print(f'summary_grid.png  -> {grid_dir / "summary_grid.png"}')
print(f'comparison_grid.png -> {grid_dir / "comparison_grid.png"}')

Saved 81 per-sample grids
summary_grid.png  -> /content/drive/MyDrive/Cobas/gan_run/results/cobas_clean_v1/test_outputs/grids/summary_grid.png
comparison_grid.png -> /content/drive/MyDrive/Cobas/gan_run/results/cobas_clean_v1/test_outputs/grids/comparison_grid.png


In [9]:
print('=' * 90)
print(f'{"Model":<22} {"Data":>14} {"HUD removed":>12} {"PSNR":>8} {"SSIM":>8} {"Notes":<20}')
print('-' * 90)
print(f'{"old epoch_0040":<22} {"old frames":>14} {"no":>12} {"25.10":>8} {"0.827":>8} {"HUD contaminated":<20}')
print(f'{best_tag + " (clean_v1)":<22} {"cleaned crops":>14} {"yes":>12} {best_res["PSNR_mean"]:>8.2f} {best_res["SSIM_mean"]:>8.4f} {"current run":<20}')
print('=' * 90)
if HAS_LPIPS:
    print(f'  LPIPS (clean_v1): {best_res["LPIPS_mean"]:.4f} +/- {best_res["LPIPS_std"]:.4f}')
print('\nNote: pixel metrics are approximate due to unpaired CycleGAN + independent crops.')
print('Visual quality (see grids) is the primary judge.')

Model                            Data  HUD removed     PSNR     SSIM Notes               
------------------------------------------------------------------------------------------
old epoch_0040             old frames           no    25.10    0.827 HUD contaminated    
epoch_0020 (clean_v1)   cleaned crops          yes    26.27   0.8766 current run         

Note: pixel metrics are approximate due to unpaired CycleGAN + independent crops.
Visual quality (see grids) is the primary judge.


In [10]:
deliverable_root = WORK / 'deliverables' / 'cobas_cycle_gan_clean_v1'
if deliverable_root.exists():
    shutil.rmtree(deliverable_root)
deliverable_root.mkdir(parents=True, exist_ok=True)

# Checkpoints
ckpt_dest = deliverable_root / 'checkpoints'
ckpt_dest.mkdir()
shutil.copy2(str(best_ckpt_path), str(ckpt_dest / f'{best_tag}.pt'))
for name in ['G_optical_to_thermal.pt', 'G_thermal_to_optical.pt']:
    p = WORK / name
    if p.exists():
        shutil.copy2(str(p), str(ckpt_dest / name))

# Test outputs
if test_out.exists():
    shutil.copytree(str(test_out), str(deliverable_root / 'test_outputs'))

# Grids
for gname in ['comparison_grid.png', 'summary_grid.png']:
    src = grid_dir / gname
    if src.exists():
        shutil.copy2(str(src), str(deliverable_root / gname))

# QA image
for qa_path in [
    Path(DRIVE_ROOT) / 'drive' / 'MyDrive' / 'images' / 'battery' / '_qa.png',
    WORK / '_qa.png',
    Path(OPTICAL_FRAMES).parent.parent.parent.parent / 'drive' / 'MyDrive' / 'images' / 'battery' / '_qa.png',
]:
    if qa_path.exists():
        shutil.copy2(str(qa_path), str(deliverable_root / 'preprocessing_qa.png'))
        break

# metrics.json
metrics_json = {
    'best_checkpoint': best_tag,
    'training_epochs': 20,
    'image_size': IMG_SIZE,
    'dataset': {
        'total_frames': N,
        'train_frames': len(train_idx),
        'test_frames': len(test_idx),
        'test_ratio': TEST_RATIO,
        'seed': SEED,
    },
    'best_metrics': best_res,
    'all_checkpoints': {k: v for k, v in sorted(all_results.items())},
    'comparison': {
        'old_epoch_0040': {'PSNR': 25.10, 'SSIM': 0.827, 'data': 'old frames', 'hud_removed': False},
        'clean_v1': {
            'PSNR': best_res['PSNR_mean'],
            'SSIM': best_res['SSIM_mean'],
            'data': 'cleaned crops',
            'hud_removed': True,
            'checkpoint': best_tag,
        },
    },
}
(deliverable_root / 'metrics.json').write_text(json.dumps(metrics_json, indent=2), encoding='utf-8')

# manifest.csv
with open(str(deliverable_root / 'manifest.csv'), 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['split', 'frame_index', 'optical_file', 'thermal_file'])
    for i in train_idx:
        w.writerow(['train', i, opt_files[i].name, thm_files[i].name])
    for i in test_idx:
        w.writerow(['test', i, opt_files[i].name, thm_files[i].name])

# README.md
lpips_line = f'  - LPIPS: {best_res["LPIPS_mean"]:.4f} +/- {best_res["LPIPS_std"]:.4f}\n' if HAS_LPIPS else ''
readme = f"""# CycleGAN Clean v1 - Optical to Thermal Translation

## Training
- Architecture: CycleGAN (ResNet-9 Generator, PatchGAN Discriminator)
- Epochs: 20
- Image size: {IMG_SIZE}x{IMG_SIZE}
- Total frames: {N} (train: {len(train_idx)}, test: {len(test_idx)})
- Device: Colab T4 GPU

## Best Checkpoint: {best_tag}
  - PSNR: {best_res['PSNR_mean']:.2f} +/- {best_res['PSNR_std']:.2f}
  - SSIM: {best_res['SSIM_mean']:.4f} +/- {best_res['SSIM_std']:.4f}
  - MAE:  {best_res['MAE_mean']:.4f}
  - RMSE: {best_res['RMSE_mean']:.4f}
  - Throughput: {best_res['throughput_img_s']:.1f} img/s
{lpips_line}
## Comparison vs Old Model
| Model | Data | HUD removed | PSNR | SSIM | Notes |
|-------|------|-------------|------|------|-------|
| old epoch_0040 | old frames | no | 25.10 | 0.827 | HUD contaminated |
| {best_tag} (clean_v1) | cleaned crops | yes | {best_res['PSNR_mean']:.2f} | {best_res['SSIM_mean']:.4f} | current run |

Pixel metrics are approximate due to unpaired CycleGAN + independent crops.
Visual quality (see comparison_grid.png) is the primary judge.
"""
(deliverable_root / 'README.md').write_text(readme, encoding='utf-8')

print(f'Packaged: {deliverable_root}')
for p in sorted(deliverable_root.rglob('*')):
    if p.is_file():
        rel = p.relative_to(deliverable_root)
        size_kb = p.stat().st_size / 1024
        print(f'  {rel}  ({size_kb:.1f} KB)')

Packaged: /content/drive/MyDrive/Cobas/gan_run/deliverables/cobas_cycle_gan_clean_v1
  README.md  (0.8 KB)
  checkpoints/G_optical_to_thermal.pt  (44437.9 KB)
  checkpoints/G_thermal_to_optical.pt  (44437.9 KB)
  checkpoints/epoch_0020.pt  (331415.0 KB)
  comparison_grid.png  (1734.3 KB)
  manifest.csv  (16.9 KB)
  metrics.json  (2.7 KB)
  summary_grid.png  (1819.8 KB)
  test_outputs/epoch_0005/000002_input.png  (23.7 KB)
  test_outputs/epoch_0005/000002_pred.png  (88.4 KB)
  test_outputs/epoch_0005/000002_target.png  (49.2 KB)
  test_outputs/epoch_0005/000009_input.png  (24.9 KB)
  test_outputs/epoch_0005/000009_pred.png  (88.3 KB)
  test_outputs/epoch_0005/000009_target.png  (50.2 KB)
  test_outputs/epoch_0005/000011_input.png  (25.5 KB)
  test_outputs/epoch_0005/000011_pred.png  (88.5 KB)
  test_outputs/epoch_0005/000011_target.png  (50.8 KB)
  test_outputs/epoch_0005/000026_input.png  (27.0 KB)
  test_outputs/epoch_0005/000026_pred.png  (88.3 KB)
  test_outputs/epoch_0005/000026_ta

In [11]:
import zipfile

zip_path = deliverable_root.parent / f'{deliverable_root.name}.zip'
with zipfile.ZipFile(str(zip_path), 'w', zipfile.ZIP_DEFLATED) as zf:
    for p in deliverable_root.rglob('*'):
        if p.is_file():
            zf.write(str(p), str(p.relative_to(deliverable_root.parent)))

size_mb = zip_path.stat().st_size / 1024 / 1024
print(f'Zipped: {zip_path}  ({size_mb:.1f} MB)')

from google.colab import files
files.download(str(zip_path))

Zipped: /content/drive/MyDrive/Cobas/gan_run/deliverables/cobas_cycle_gan_clean_v1.zip  (456.2 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>